# Experimentation with the readings of the signal code

In [5]:
import os
import numpy as np
from matplotlib import pyplot as plt
from Functions.Functions_placeholdername import read_multichannel_bin_data



## Example 1 : chargement du début de fichier, seulement 10s 





In [ ]:
filepath = r"Data\Donnes\Old_Data\20180116-MEAPedot8-P1107-MERGE.bin"
skip_s   = 0
length_s = 0

Fs = 14656 #14656 10000
data = read_multichannel_bin_data(filepath, length_s=length_s, skip_s=skip_s)

L = data.shape[0] # Data length (samples)
N = data.shape[1] # Number of channels

time = np.linspace(start=skip_s, stop=skip_s + L/Fs, num=L, endpoint=False)

Reading 7293312000B from offset 0 (0x0)
20 first points of data (binary values) : 
0,-384,-335,-1013,-870,-750,-281,0,-1593,71,-277,-705,-536,-1290,245,-170,-1165,-1174,-1209,-1165,
All done.


In [ ]:
print(f"Data shape : {data.shape}")
print(data)
print(f"Time shape : {time.shape}")
print(time)

E = 55 # Channel of interest (idx 20 <=> electrode 35)
plt.figure(figsize=(15,5))
plt.plot(time, data[:,E])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [µV]")
plt.title(f"Channel #{E}")
plt.plot()
plt.show()

"""E = 48 # Channel of interest (idx 48 <=> electrode 71)
plt.figure(figsize=(15,5))
plt.plot(time, data[:,E])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [µV]")
plt.title(f"Channel #{E}")
plt.plot()
plt.show()"""

"""E = 15 # Channel of interest (idx 15 <=> electrode 28)
plt.figure(figsize=(15,5))
plt.plot(time, data[:,E])
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [µV]")
plt.title(f"Channel #{E}")
plt.plot()
plt.show()"""


Data shape : (56979000, 64)
[[  0.         -1.9531392  -1.7039105 ...   2.3142665  -0.6815642
    0.       ]
 [  0.         -5.493204    1.0223463 ...   7.0394392  -5.8543313
    0.       ]
 [  0.         -8.036354   -8.4229128 ...   0.864671   -8.2194608
    0.       ]
 ...
 [  0.         -2.6804801  -5.3355287 ... -12.0290995  -4.7658631
    0.       ]
 [  0.          0.0457767  -6.3324435 ...  -6.2205449  -1.8615858
    0.       ]
 [  0.         -2.2227131  -5.3355287 ...  -6.8563324  -0.406904
    0.       ]]
Time shape : (56979000,)
[0.00000000e+00 6.82314410e-05 1.36462882e-04 ... 3.88775907e+03
 3.88775914e+03 3.88775921e+03]


## Informations sur le fichier 

In [16]:
def read_multichannel_bin_info(filepath):
    Fs               = 10000 # Hz
    N_channels       = 64 # Channnels
    data_width_bytes = 2  # 2 bytes per sample (>i2 : signed 2-byte integers, big-endian)
    filesize = os.path.getsize(filepath)
    
    # At every sampling time (1/Fs), 64 samples (1 for every channel) are recorded
    # They are recorded in a 128B chunk (2B per channel)
    #   @0x0000 <Sample0/Ch0><Sample0/Ch1><Sample0/Ch2> ... <Sample0/Ch63> (@t=0 s)
    #   @0x0080 <Sample1/Ch0><Sample1/Ch1><Sample1/Ch2> ... <Sample1/Ch63> (@t=0.1 ms)
    #   @0x0100 <Sample2/Ch0><Sample2/Ch1><Sample2/Ch2> ... <Sample2/Ch63> (@t=0.2 ms)
    #                                                   ...
    #   @0x---- <SampleN/Ch0><SampleN/Ch1><SampleN/Ch2> ... <SampleN/Ch63> (@t=N*0.1e-3 s)

    length_chunks = int(filesize / (N_channels * data_width_bytes))
    length_bytes = int(length_chunks * N_channels * data_width_bytes)
    length_samples = length_bytes // data_width_bytes
    
    length_s = length_samples
    bitrate_Bps = N_channels * data_width_bytes * Fs
    
    print(f"File is {length_bytes} bytes long.")
    print(f"Bitrate is {bitrate_Bps} B/s")
    print(f"File is {length_bytes / bitrate_Bps} s long.")
    print(f"Expected data size is {length_chunks}x{N_channels}")
    
read_multichannel_bin_info(filepath)

File is 7293312000 bytes long.
Bitrate is 1280000 B/s
File is 5697.9 s long.
Expected data size is 56979000x64


In [ ]:



def read_multichannel_bin_data_separate_save(filepath, quantum=0.0050863):
    # filepath : path to the .bin file
    # quantum  : acquisition quantum -- scaling factor that converts the raw binary values (signed 16-bit integers, >i2) into physical units (likely volts or microvolts).
        #The file contains 2-byte signed integers representing the raw ADC (Analog-to-Digital Converter) counts from the acquisition hardware.

    # skip_s   : Time (in seconds) to skip (ie. ignore) at the beginning of the file
    # length_s : Time (in seconds) of data to read (leave 0 to read whole file)
    
    Fs               = 10000 # Hz
    N_channels       = 64 # Channnels -- electrodes, each channel is an electrode in the MEA
    data_width_bytes = 2  # 2 bytes per sample (>i2 : signed 2-byte integers, big-endian)
    filesize = os.path.getsize(filepath)
    
    # At every sampling time (1/Fs), 64 samples (1 for every channel) are recorded
    # They are recorded in a 128B chunk (2B per channel)
    #   @0x0000 <Sample0/Ch0><Sample0/Ch1><Sample0/Ch2> ... <Sample0/Ch63> (@t=0 s)
    #   @0x0080 <Sample1/Ch0><Sample1/Ch1><Sample1/Ch2> ... <Sample1/Ch63> (@t=0.1 ms)
    #   @0x0100 <Sample2/Ch0><Sample2/Ch1><Sample2/Ch2> ... <Sample2/Ch63> (@t=0.2 ms)
    #                                                   ...
    #   @0x---- <SampleN/Ch0><SampleN/Ch1><SampleN/Ch2> ... <SampleN/Ch63> (@t=N*0.1e-3 s)

    
    length_chunks = int(filesize / (N_channels * data_width_bytes))
    ''' Number of bytes in requested data '''
    skip_bytes   = int(N_channels * data_width_bytes)
    length_bytes = int(N_channels * data_width_bytes)
    
    print(f"Reading {length_bytes}B from offset {skip_bytes} (0x{skip_bytes:X})")
    
    ''' Read file'''
    with open(filepath, 'rb') as fid:
        if skip_bytes > 0:
            fid.seek(skip_bytes)
        data = np.fromfile(fid, dtype='>i2', count=length_bytes // data_width_bytes)
    print("20 first points of data (binary values) : ")
    for x in data[:20]:
        print(x, end=',')
    print()
    
    #data = data.reshape(length_chunks, N_channels)
    os.makedirs(r"Data\Recordings_per_channel", exist_ok= True)
    output_dir = r"Data\Recordings_per_channel"
    
    for channel in len(data.shape[1]):
        x = data[:][channel]
        np.savetxt(f"{output_dir}/channel_{channel+1}.txt")

    print("All done.")


filepath = r"Data\Donnes\donnees\20180116-MEAPedot8-P1107-MERGE.bin"
skip_s   = 0
length_s = 0

#Fs = 14656 #14656 10000
read_multichannel_bin_data_separate_save(filepath, quantum=0.0050863)
    


In [ ]:
import os
import numpy as np
def read_and_save_channels(filepath, quantum=0.0050863):
    """
    Lee un archivo .bin multicanal y guarda cada canal como un archivo .npy independiente.
    
    Args:
        filepath (str): Ruta al archivo .bin
        output_dir (str): Directorio donde se guardarán los canales
        quantum (float): Factor de conversión a unidades físicas
    
    Returns:
        dict: Diccionario con los datos de cada canal (opcional)
    """
    
    # Parámetros fijos del sistema
    Fs = 10000  # Hz
    N_channels = 64  # Número de canales
    data_width_bytes = 2  # 2 bytes por muestra
    
    # Obtener tamaño del archivo
    filesize = os.path.getsize(filepath)
    
    # Calcular número total de muestras por canal
    samples_per_channel = filesize // (N_channels * data_width_bytes)
    
    print(f"Archivo: {filepath}")
    print(f"Tamaño total: {filesize} bytes")
    print(f"Muestras por canal: {samples_per_channel}")
    print(f"Duración total: {samples_per_channel / Fs:.2f} segundos")
    
    # Leer todo el archivo
    print("Leyendo archivo...")
    with open(filepath, 'rb') as fid:
        data = np.fromfile(fid, dtype='>i2', count=samples_per_channel * N_channels)
    
    print(f"Datos leídos: {len(data)} valores")
    
    # Reorganizar en matriz (tiempo, canales)
    print("Reorganizando datos...")
    data_matrix = data.reshape(samples_per_channel, N_channels)
    
    # Aplicar quantum (conversión a unidades físicas)
    data_matrix = data_matrix * quantum
    
    # Crear directorio de salida si no existe
    os.makedirs(r"Data\Recordings_per_channel", exist_ok= True)
    output_dir = r"Data\Recordings_per_channel"
    print(f"Guardando en: {output_dir}")
    
    # Guardar cada canal por separado
    print("Guardando canales...")
    channel_data = {}
    
    for i in range(N_channels):
        # Extraer canal
        channel = data_matrix[:, i]
        
        # Guardar como archivo .npy
        #np.savetxt(f"{output_dir}/channel_{channel+1}.txt")
        filename = os.path.join(output_dir,f"channel_{i+1:02d}.npy")
        #filename = output_dir / f"channel_{i+1:02d}.txt"
        np.save(filename, channel)
        
        # Guardar referencia en diccionario
        channel_data[f"ch{i+1}"] = channel
        
        # Mostrar progreso cada 10 canales
        if (i + 1) % 10 == 0:
            print(f"  Guardados {i+1}/{N_channels} canales")
    
    print(f"Completado. {N_channels} canales guardados en {output_dir}")

filepath = r"Data\Donnes\donnees\20180116-MEAPedot8-P1107-MERGE.bin"

read_and_save_channels(filepath, quantum=0.0050863)


Archivo: Data\Donnes\donnees\20180116-MEAPedot8-P1107-MERGE.bin
Tamaño total: 7293312000 bytes
Muestras por canal: 56979000
Duración total: 5697.90 segundos
Leyendo archivo...
Datos leídos: 3646656000 valores
Reorganizando datos...
Guardando en: Data\Recordings_per_channel
Guardando canales...
